In [12]:
import pandas as pd
import numpy as np
from scipy.stats import wasserstein_distance, ks_2samp, skew, kurtosis
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [3]:
# ==============================================================================
# 1. LOAD LONG FORMAT PREDICTIONS
# ==============================================================================
print("Loading long-format TimesFM predictions...")
df_long = pd.read_parquet('../data/processed/timesfm_predictions_long_production.parquet')

# Calcular el vector de Residuos (Error) nativamente en pandas
df_long['residual'] = df_long['value'] - df_long['value_timesfm']

print(f"Dataset loaded. Total points: {len(df_long)}")

Loading long-format TimesFM predictions...
Dataset loaded. Total points: 6246629


In [ ]:
# ==============================================================================
# 2. CUSTOM TOPOLOGICAL & DISTRIBUTIONAL FEATURES
# ==============================================================================
print("Calculating advanced distribution and topology features (Domain-Agnostic)...")

unique_ids = df_long['id'].unique()
custom_features = []

errors_printed = 0 

for uid in tqdm(unique_ids, desc="Custom Features"):
    group = df_long[df_long['id'] == uid]
    
    # Base dictionary for safety
    feat_dict = {
        'id': uid, 'tfm_dtw_dist': np.nan, 'tfm_wasserstein': np.nan, 
        'tfm_ks_stat': np.nan, 'tfm_skew_diff': np.nan, 'tfm_kurt_diff': np.nan
    }
    
    actual = group['value'].values
    pred = group['value_timesfm'].values
    
    if len(actual) < 2:
        custom_features.append(feat_dict)
        continue
        
    try:
        # 1. Dynamic Time Warping (DTW) - Shape Alignment
        # FIX: Removed dist=euclidean. fastdtw uses absolute diff natively for 1D arrays.
        distance, _ = fastdtw(actual, pred)
        feat_dict['tfm_dtw_dist'] = distance
        
        # 2. Wasserstein Distance (Earth Mover's Distance)
        feat_dict['tfm_wasserstein'] = wasserstein_distance(actual, pred)
        
        # 3. Kolmogorov-Smirnov Test Statistic
        feat_dict['tfm_ks_stat'] = ks_2samp(actual, pred).statistic
        
        # 4. Tails and Asymmetry (Skewness & Kurtosis Differences)
        feat_dict['tfm_skew_diff'] = skew(actual) - skew(pred)
        feat_dict['tfm_kurt_diff'] = kurtosis(actual) - kurtosis(pred)
        
    except Exception as e:
        # Fail fast visibility: print the first 3 errors to know what's wrong
        if errors_printed < 3:
            print(f"\n[DEBUG] Error on ID {uid}: {type(e).__name__} - {e}")
            errors_printed += 1
        pass
        
    custom_features.append(feat_dict)

df_custom = pd.DataFrame(custom_features).set_index('id')

print("\nQuick check of the generated features:")
display(df_custom.head())

Calculating advanced distribution and topology features (Domain-Agnostic)...


Custom Features: 100%|██████████| 10001/10001 [05:16<00:00, 31.55it/s]


Quick check of the generated features:


,tfm_dtw_dist,tfm_wasserstein,tfm_ks_stat,tfm_skew_diff,tfm_kurt_diff
id,,,,,
0,1.617093,0.005325,0.472789,-0.314878,-1.101128
1,0.449395,0.001409,0.471631,-0.597267,8.092524
2,8.527915,0.015939,0.524272,1.183165,4.740338
3,4.678785,0.006704,0.492846,2.483299,-0.138397
4,1.233579,0.002630,0.453947,7.644798,-81.945870


In [13]:
# ==============================================================================
# 3. TSFRESH ON RESIDUALS
# ==============================================================================
print("\nExtracting features from the Residuals using tsfresh...")

# We only pass the ID, Time, and Residual to tsfresh
df_residuals = df_long[['id', 'time', 'residual']]

df_tsfresh_res = extract_features(
    df_residuals,
    column_id='id',
    column_sort='time',
    default_fc_parameters=EfficientFCParameters(),
    disable_progressbar=False
)

# Rename columns to clearly identify them as TimesFM Residual features
df_tsfresh_res.columns = [f"tfm_res_{col}" for col in df_tsfresh_res.columns]


Extracting physical and entropy signatures from the Residuals using tsfresh...


Feature Extraction: 100%|██████████| 30/30 [03:47<00:00,  7.59s/it]


In [14]:
# ==============================================================================
# 4. MERGE, IMPUTE & SAVE
# ==============================================================================
print("\nMerging all Meta-Features...")

# Join the custom features with the tsfresh residual features
df_final_tfm = df_custom.join(df_tsfresh_res)

# Impute any NaNs (from failed series or impossible math like entropy on 1 data point)
# using the median, preserving the integrity of downstream models
df_final_tfm = df_final_tfm.fillna(df_final_tfm.median())

# Save to Parquet for the final Optuna notebook
save_path = '../data/processed/features_timesfm.parquet'
df_final_tfm.to_parquet(save_path)

print(f"\nAdvanced TimesFM meta-features successfully saved to '{save_path}'!")
print("\nSnapshot of the new features:")
display(df_final_tfm.head())


Merging all Meta-Features...

Advanced TimesFM meta-features successfully saved to '../data/processed/features_timesfm.parquet'!

Snapshot of the new features:


,tfm_dtw_dist,tfm_wasserstein,tfm_ks_stat,tfm_skew_diff,tfm_kurt_diff,tfm_res_residual__variance_larger_than_standard_deviation,tfm_res_residual__has_duplicate_max,tfm_res_residual__has_duplicate_min,tfm_res_residual__has_duplicate,tfm_res_residual__sum_values,...,tfm_res_residual__fourier_entropy__bins_5,tfm_res_residual__fourier_entropy__bins_10,tfm_res_residual__fourier_entropy__bins_100,tfm_res_residual__permutation_entropy__dimension_3__tau_1,tfm_res_residual__permutation_entropy__dimension_4__tau_1,tfm_res_residual__permutation_entropy__dimension_5__tau_1,tfm_res_residual__permutation_entropy__dimension_6__tau_1,tfm_res_residual__permutation_entropy__dimension_7__tau_1,tfm_res_residual__query_similarity_count__query_None__threshold_0.0,tfm_res_residual__mean_n_absolute_max__number_of_maxima_7
id,,,,,,,,,,,,,,,,,,,,,
0,1.617093,0.005325,0.472789,-0.314878,-1.101128,0.0,0.0,0.0,0.0,-0.051740,...,0.795727,1.372118,3.375872,1.784567,3.149441,4.549963,5.371914,5.588941,NaN,0.017065
1,0.449395,0.001409,0.471631,-0.597267,8.092524,0.0,0.0,0.0,0.0,-0.055167,...,1.142227,1.795089,3.714189,1.791100,3.138579,4.516201,5.347436,5.575196,NaN,0.006752
2,8.527915,0.015939,0.524272,1.183165,4.740338,0.0,0.0,0.0,0.0,-1.066574,...,1.042582,1.695340,3.718173,1.788947,3.158000,4.661089,5.792207,6.183424,NaN,0.094382
3,4.678785,0.006704,0.492846,2.483299,-0.138397,0.0,0.0,0.0,0.0,-0.549350,...,1.269848,1.936313,3.820782,1.789096,3.160215,4.701202,5.944159,6.339114,NaN,0.038338
4,1.233579,0.002630,0.453947,7.644798,-81.945870,0.0,0.0,0.0,0.0,-0.046722,...,0.620804,1.087285,3.088413,1.770542,3.089131,4.527859,5.656714,6.029906,NaN,0.008965
